In [3]:
from langchain_openai import ChatOpenAI 
from langchain_groq import ChatGroq
from langchain.document_loaders import  PyPDFLoader
from langchain.vectorstores import  FAISS
from langchain.text_splitter import  RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document
from langchain.chains.summarize import load_summarize_chain
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables.graph import MermaidDrawMethod

from langgraph.graph import END, StateGraph

from time import monotonic
from dotenv import load_dotenv
from pprint import pprint
import os
from datasets import Dataset
from typing_extensions import TypedDict
from IPython.display import display, Image
from typing import TypedDict, Literal, Optional, List

from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_similarity
)

import langgraph


load_dotenv(override=True)

os.environ["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "100000"

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env

# 用 DeepSeek 的 key 伪装成 OPENAI_API_KEY，让 ChatOpenAI 觉得自己有 key
os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com"


In [ ]:
IntentType = Literal["faq", "tech_issue", "account", "other"]

class SupportState(TypedDict, total=False):
    """TechSupport-Agent 在 graph 里的共享状态."""
    query: str                           # 当前用户问题
    intent: IntentType                   # 意图分类结果
    answer: str                          # 当前节点生成的回答（最后可以用于输出）
    retrieved_docs: List[Document]       # 如果有检索，就放这里（目前只有 FAQ 用）

# 数据准备

In [6]:
#解析出正文
import os
from typing import List, Dict, Tuple
import yaml
from pathlib import Path

# 兼容：脚本文件运行 / notebook / 交互环境
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
else:
    # 在 notebook 或交互式环境下，就用当前工作目录作为项目根目录
    PROJECT_ROOT = Path(os.getcwd())

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTORDDB_DIR = PROJECT_ROOT / "vectordb"


def parse_frontmatter(text: str) -> Tuple[Dict, str]:
    """
    解析以 --- 开头的 frontmatter，返回 (metadata, body_text)
    如果没有 frontmatter，就返回 ({}, 原文)
    """
    text = text.lstrip("\ufeff")  # 去掉 BOM
    if not text.startswith("---"):
        return {}, text

    parts = text.split("---", 2)
    if len(parts) < 3:
        return {}, text

    fm_text = parts[1]
    body = parts[2].lstrip("\n")
    try:
        metadata = yaml.safe_load(fm_text) or {}
    except Exception:
        metadata = {}

    return metadata, body

In [7]:
#创建某个域的document
def load_domain_documents(domain: str) -> List[Document]:
    """
    domain: 'faq' / 'tech' / 'account'
    从 data/processed/{domain} 读取所有 .md，解析 frontmatter，
    构造 LangChain Document 列表。
    """
    domain_dir = os.path.join(PROCESSED_DIR, domain)
    docs: List[Document] = []

    for name in os.listdir(domain_dir):
        if not name.endswith(".md"):
            continue

        path = os.path.join(domain_dir, name)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        metadata, body = parse_frontmatter(text)
        # 补一些通用 metadata
        metadata = metadata or {}
        metadata.setdefault("domain", domain)
        metadata.setdefault("source_path", path)

        docs.append(Document(page_content=body, metadata=metadata))

    return docs


In [8]:
#切块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)


def split_domain_documents(domain: str) -> List[Document]:
    raw_docs = load_domain_documents(domain)
    chunks = text_splitter.split_documents(raw_docs)
    # 可以顺手在 metadata 里标记 chunk_id 之类的
    for i, doc in enumerate(chunks):
        doc.metadata.setdefault("chunk_id", i)
    return chunks

#### 向量库构建（简易版--三个共用一套）

In [9]:
pdf_path ="deepseek.pdf"

loader = PyPDFLoader(pdf_path)
raw_docs = loader.load()


len(raw_docs), raw_docs[0][:200] if isinstance(raw_docs[0], str) else raw_docs[0]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     # 每块大概 800 字符
    chunk_overlap=100,  # 相邻块有一点重叠，避免句子被硬拆开
)

docs = text_splitter.split_documents(raw_docs)

len(docs), docs[0]

def replace_t_with_space(list_of_documents):
    for doc in list_of_documents:
        doc.page_content = doc.page_content.replace('\t', ' ')  # Replace tabs with spaces
    return list_of_documents

def encode_book(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using HuggingFace embeddings.
    """
    loader = PyPDFLoader(path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore


In [10]:
faq_vectorstore = encode_book(pdf_path)
faq_retriever = faq_vectorstore.as_retriever(search_kwargs={"k": 4})

faq_vectorstore, faq_retriever

(<langchain_community.vectorstores.faiss.FAISS at 0x7f0818eac320>,
 VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f0818eac320>, search_kwargs={'k': 4}))

# 向量库构建

In [4]:
#嵌入模型
def get_embeddings():
    # 第一次会自动下载模型，注意环境要能连 huggingface 或你提前下好
    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-zh-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    )

embeddings = get_embeddings()  

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def build_and_save_vectorstore_for_domain(domain: str):
    os.makedirs(VECTORDDB_DIR, exist_ok=True)

    print(f"[{domain}] 加载并切分文档...")
    docs = split_domain_documents(domain)
    print(f"[{domain}] 共 {len(docs)} 个 chunks")

    print(f"[{domain}] 构建 FAISS 向量库...")
    vectordb = FAISS.from_documents(docs, embeddings)

    save_dir = os.path.join(VECTORDDB_DIR, domain)
    os.makedirs(save_dir, exist_ok=True)
    vectordb.save_local(save_dir)
    print(f"[{domain}] 向量库已保存到 {save_dir}")

In [12]:
#三个创建好
def build_all_vectorstores():
    for domain in ["faq", "tech", "account"]:
        build_and_save_vectorstore_for_domain(domain)


if __name__ == "__main__":
    build_all_vectorstores()

[faq] 加载并切分文档...
[faq] 共 56 个 chunks
[faq] 构建 FAISS 向量库...
[faq] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/faq
[tech] 加载并切分文档...
[tech] 共 38 个 chunks
[tech] 构建 FAISS 向量库...
[tech] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/tech
[account] 加载并切分文档...
[account] 共 9 个 chunks
[account] 构建 FAISS 向量库...
[account] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/account


##### test

In [16]:
def load_vectorstore(domain: str) -> FAISS:
    vs_path = VECTORDDB_DIR / domain
    if not vs_path.exists():
        raise FileNotFoundError(f"向量库目录不存在：{vs_path}")
    embeddings = get_embeddings()
    vectordb = FAISS.load_local(
        str(vs_path),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    return vectordb

def pretty_print_results(domain: str, query: str, k: int = 3):
    print("=" * 80)
    print(f"[{domain.upper()}] query = {query}")
    vectordb = load_vectorstore(domain)
    docs_scores = vectordb.similarity_search_with_score(query, k=k)

    if not docs_scores:
        print("  （没有检索到结果）")
        return

    for i, (doc, score) in enumerate(docs_scores, 1):
        meta = doc.metadata or {}
        section = meta.get("section", "N/A")
        source_url = meta.get("source_url", "N/A")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"\n  Top {i}: score={score:.4f}")
        print(f"    section: {section}")
        print(f"    source_url: {source_url}")
        print(f"    preview: {preview}...")

# 1) 限速相关 -> 预期 hits tech/ rate_limit
q_rate = "接口的限速规则是怎样的？QPS 和并发有什么限制？"
pretty_print_results("tech", q_rate, k=3)

# 2) 价格/计费 -> 预期 hits account/ pricing, token_usage
q_price = "deepseek-chat 每百万 tokens 多少钱？硬盘缓存是怎么算费用的？"
pretty_print_results("account", q_price, k=3)

# 3) 使用方式 -> 预期 hits faq/ first_api_call
q_faq = "我应该怎么用 Python 调用 DeepSeek 的对话 API？"
pretty_print_results("faq", q_faq, k=3)


[TECH] query = 接口的限速规则是怎样的？QPS 和并发有什么限制？

  Top 1: score=0.8973
    section: rate_limit
    source_url: https://api-docs.deepseek.com/zh-cn/quick_start/rate_limit
    preview: # 限速  DeepSeek API **不限制用户并发量**，我们会尽力保证您所有请求的服务质量。  但请注意，当我们的服务器承受高流量压力时，您的请求发出后，可能需要等待一段时间才能获取服务器的响应。在这段时间里，您的 HTTP 请求会保持连接，并持续收到如下格式的返回内容：  * 非流式请求：持续返回空行 * 流...

  Top 2: score=1.0566
    section: create_chat_completion
    source_url: https://api-docs.deepseek.com/zh-cn/api/create-chat-completion
    preview: * Array [  **token** stringrequired  输出的 token。  **logprob** numberrequired  该 token 的对数概率。`-9999.0` 代表该 token 的输出概率极小，不在 top 20 最可能输出的 token 中。  **bytes** inte...

  Top 3: score=1.0680
    section: create_chat_completion
    source_url: https://api-docs.deepseek.com/zh-cn/api/create-chat-completion
    preview: "prompt_tokens": 0,     "prompt_cache_hit_tokens": 0,     "prompt_cache_miss_tokens": 0,     "total_tokens": 0,     "completion_tokens_details": {       "reason...
[ACCOUNT] query = deepseek-cha

# 构建IntentClassifier链

In [ ]:
class IntentResult(BaseModel):
    intent: IntentType = Field(
        description="用户问题的意图，必须是 'faq', 'tech_issue', 'account', 或 'other' 之一。"
    )
    reason: str = Field(
        description="用简短中文解释为什么做出这个意图判断。"
    )

In [ ]:
intent_prompt_template="""
你是一个技术支持平台的意图分类助手，需要把用户的问题归类到以下四类之一：

1. faq：纯文档/说明类问题，常见模式：
   - “这个接口怎么用？”
   - “某个参数是什么意思？”
   - “错误码 401 的含义是什么？”
   重点在于查文档就能回答的说明类问题。

2. tech_issue：环境 / 报错 / 排错类问题，常见模式：
   - “我调用接口报 401/429/5xx 怎么办？”
   - “向量数据库连接失败 connection refused？”
   - “docker compose 启动某个服务失败？”
   重点在于“出错了，需要一步步排查”。

3. account：账号 / 计费 / 配额 / 限速咨询，常见模式：
   - “免费额度用完会发生什么？”
   - “为什么提示 quota exceeded？”
   - “这个 key 有没有开通某个权限？”
   重点在于账号状态、套餐、配额、计费策略等。

4. other：不符合以上三类的其他问题，或者难以判断的情况。

【用户问题】
{query}

请根据问题内容，给出 intent（faq / tech_issue / account / other）以及简短的 reason。
""".strip()

intent_prompt = PromptTemplate(
    template=intent_prompt_template,
    input_variables=["query"],
)

intent_llm = ChatOpenAI(
    temperature=0, model_name="deepseek-chat", max_tokens=512,
)

intent_chain = intent_prompt | intent_llm.with_structured_output(IntentResult)

In [ ]:
def intent_classifier_node(state: SupportState) -> SupportState:
    """根据 state['query'] 判断意图，并写回 state['intent']。"""
    query = state["query"]
    result: IntentResult = intent_chain.invoke({"query": query})
    state["intent"] = result.intent
    # 如果你后面想在调试时看看 reason，也可以临时 print 一下：
    # print("[Intent]", result.intent, "| reason:", result.reason)
    return state

#### test

In [ ]:
tests = [
    "如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？",
    "我调用 API 一直报 401 unauthorized，怎么办？",
    "免费额度用完之后接口还能用吗？",
    "你觉得大模型会统治世界吗？",
]

for q in tests:
    r = intent_chain.invoke({"query": q})
    print("Q:", q)
    print("intent:", r.intent)
    print("reason:", r.reason)
    print("-" * 60)


### 构建FAQ链

In [ ]:
class FAQAnswer(BaseModel):
    """FAQ 最终答案的结构化输出。"""
    short_answer: str = Field(
        description="用 1-3 句话直接回答用户问题的核心结论。"
    )
    details: str = Field(
        description="更详细的说明，可以包含参数解释、使用建议等。"
    )
    #caveats: Optional[str] = Field(
    #    default=None,
    #    description="如果文档中有未说明的点，或者需要提示用户注意的地方，在这里说明。",
    #)


In [ ]:
faq_answer_prompt_template = """
你是一名熟悉 DeepSeek API 文档的技术支持工程师。

【用户问题】
{question}

【与问题相关的文档内容】（已经经过预处理，只保留了和问题强相关的部分）
{relevant_content}

请严格基于上述文档内容回答用户的问题，不要编造文档中没有的信息。
回答时遵循以下要求：

1. short_answer：用 1-3 句中文直接回答用户问题的核心结论。
2. details：用一段或几段话，详细说明相关接口/参数/错误码的含义和使用方式，可以使用列表或分点说明。
3. caveats：如果文档中没有包含用户想问的某些信息，或者有需要特别提醒用户注意的地方（例如限速、权限、配额等），在这里简要说明；如果没有，可以写“无”或留空。

请按上述字段生成结构化输出。
""".strip()

faq_answer_prompt = PromptTemplate(
    template=faq_answer_prompt_template,
    input_variables=["question", "relevant_content"],
)

faq_answer_llm = ChatOpenAI(
    temperature=0, model_name="deepseek-chat", max_tokens=2000
)

faq_answer_chain = faq_answer_prompt | faq_answer_llm.with_structured_output(FAQAnswer)

In [ ]:
def answer_faq(question: str, top_k: int = 4) -> FAQAnswer:
    """
    FAQ v1（structured_output 版）：
    1. 用 faq_retriever 检索 top_k 段文本
    2. 拼成 context
    3. 调用 faq_structured_chain，返回 FAQAnswer 对象
    """
    # 1. 检索知文档
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    # 2. 粗暴拼接上下文（先不做“只保留相关内容”的中间层）
    context = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    # 3. 调用 structured_output 链
    result: FAQAnswer = faq_answer_chain.invoke(
        {"question": question, "relevant_content": context}
    )

    return result

In [ ]:
def faq_node(state: SupportState) -> SupportState:
    question = state["query"]          # 这里把 state 的 query 映射给 answer_faq 的 question
    faq_result = answer_faq(question)  # 调用你上面的链

    final_answer = (
        f"【FAQ 简要回答】\n{faq_result.short_answer}\n\n"
        f"【FAQ 详细说明】\n{faq_result.details}"
    )
    if faq_result.caveats and faq_result.caveats.strip() and faq_result.caveats.strip() != "无":
        final_answer += f"\n\n【注意事项】\n{faq_result.caveats}"

    state["answer"] = final_answer
    return state

#### test

In [ ]:
q1 = "如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？"
q2 = "temperature 参数是干什么用的？一般怎么设置？"

for q in [q1, q2]:
    print("=" * 80)
    print("问题：", q)
    ans = answer_faq(q)
    print("\n[short_answer]")
    print(ans.short_answer)
    print("\n[details]")
    print(ans.details)
    print("\n\n")


### 构建techissue链

In [ ]:
class TechIssueAnswer(BaseModel):
    """技术问题（报错/环境）的结构化排错输出。"""
    summary: str = Field(
        description="用 1-3 句中文概括问题本质和大致方向，例如：'这是一个认证失败相关的问题'"
    )
    possible_causes: List[str] = Field(
        description="可能的原因列表，每条是一句话或一小段，例如配置错误、权限不足、网络问题等"
    )
    steps: List[str] = Field(
        description="推荐的排查步骤列表，按顺序执行（Step 1, Step 2...），每条一步"
    )

In [ ]:
tech_issue_prompt_template = """
你是一名熟悉 DeepSeek API 文档的技术支持工程师，专门帮助用户排查“调用出错 / 环境问题”。

【用户问题】
{question}

【与问题相关的文档内容】（来自官方文档，可能包含错误码说明、限速说明、接口用法示例等）
{relevant_content}

请严格基于上述文档内容进行分析，不要编造文档中没有的信息。
请按以下结构化方式输出排查建议（对应 TechIssueAnswer 模型）：

1. summary：
   - 用 1-3 句中文概括这个问题大概是哪一类（例如认证失败、配额耗尽、限速、请求格式错误等）。
   - 如果文档中没有足够信息确定具体原因，请使用“可能是……，需要进一步确认”的语气。

2. possible_causes：
   - 输出一个列表，每一项是一个“可能的原因”，例如：
     - API Key 无效或没有对应权限
     - 请求中 model 字段填写错误
     - 触发了限速或配额限制
   - 这些原因必须能够从文档内容推断出来，或者是文档中明确提到的常见场景。

3. steps：
   - 输出一个“按顺序排查的步骤”列表，例如：
     - 第一步：在控制台确认 API Key 是否有效
     - 第二步：确认请求头 Authorization 是否正确设置
     - 第三步：检查请求体中 model / messages / content 是否符合文档要求
   - 每条步骤尽量具体，可直接给用户操作建议。

如果文档中完全没有提到与该问题相关的信息，请在 summary 中说明这一点，
并在 possible_causes 和 steps 中给出通用的、保守的建议（例如检查网络、检查 key、查看控制台日志等）。
""".strip()

tech_issue_prompt = PromptTemplate(
    template=tech_issue_prompt_template,
    input_variables=["question", "relevant_content"],
)

tech_issue_llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    max_tokens=2000,
)

tech_issue_chain = tech_issue_prompt | tech_issue_llm.with_structured_output(TechIssueAnswer)

In [ ]:
def answer_tech_issue(question: str, top_k: int = 4) -> TechIssueAnswer:
    """
    Tech Issue v1（structured_output 版）：
    1. 用（暂时复用的）faq_retriever 检索 top_k 段文本
    2. 拼成 relevant_content
    3. 调用 tech_issue_chain，返回 TechIssueAnswer 对象
    """
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    relevant_content = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    result: TechIssueAnswer = tech_issue_chain.invoke(
        {
            "question": question,
            "relevant_content": relevant_content,
        }
    )

    return result

In [ ]:
def tech_issue_node(state: SupportState) -> SupportState:
    """Tech issue 处理节点：用 TechIssueAnswer 结构输出排错建议。"""
    question = state["query"]
    issue_result = answer_tech_issue(question)

    # 格式化为最终展示文本（以后可以在 UI 里直接用结构化字段）
    parts = [
        f"【问题概述】\n{issue_result.summary}",
        "\n【可能原因】",
    ]
    for i, cause in enumerate(issue_result.possible_causes, start=1):
        parts.append(f"{i}. {cause}")
    parts.append("\n【建议排查步骤】")
    for i, step in enumerate(issue_result.steps, start=1):
        parts.append(f"{i}. {step}")

    state["answer"] = "\n".join(parts)
    return state

### account链

In [ ]:
class AccountAnswer(BaseModel):
    """
    账号 / 计费 / 配额 类问题的结构化输出。
    """
    short_answer: str = Field(
        description="用 1-3 句中文，直接说明用户问题的核心结论。"
    )
    details: str = Field(
        description="更详细的说明，包括计费规则、配额/限速机制、常见情形等。"
    )
    next_steps: Optional[str] = Field(
        default=None,
        description="给用户的后续操作建议，例如去控制台哪里查看用量、如何更换套餐、如何联系客服等。"
    )


In [ ]:
account_answer_prompt_template = """
你是一名熟悉 DeepSeek 平台计费与配额规则的客服工程师。

【用户问题】
{question}

【与问题相关的文档内容】（来自官方文档，可能包含计费说明、配额规则、限速策略等）
{relevant_content}

请严格基于上述文档内容回答用户的问题，不要编造文档中没有的信息。
回答时遵循以下要求（对应 AccountAnswer 模型的字段）：

1. short_answer：
   - 用 1-3 句中文，直接说明用户问题的核心结论。
   - 例如：说明“免费额度用完后，接口会返回配额耗尽错误；需要升级套餐或等待重置”。

2. details：
   - 更详细地解释相关规则，可以包括：
     - 计费方式（按调用量、按 token、按模型等）
     - 免费额度 / 试用额度的限制
     - 配额耗尽或限速时的典型错误码与表现
   - 可以使用列表或分点说明，但仍然要基于文档内容。

3. next_steps：
   - 给出用户可以执行的后续操作建议，例如：
     - 在控制台某个页面查看用量和账单
     - 确认当前 API Key 是否有对应权限
     - 考虑升级套餐、开通付费、或者联系人工客服
   - 如果文档中没有提供明确建议，可以给出通用的、保守的建议；如果实在没法建议，可以写“无”。

注意：
- 你无法访问用户的真实账号、用量或订单信息，只能给出通用说明和建议。
- 如果文档中没有包含用户关心的某个细节，请在 details 中说明“文档中未提到这一点”，不要编造具体数值或政策。
""".strip()

account_answer_prompt = PromptTemplate(
    template=account_answer_prompt_template,
    input_variables=["question", "relevant_content"],
)

account_answer_llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    max_tokens=2000,
)

account_answer_chain = account_answer_prompt | account_answer_llm.with_structured_output(AccountAnswer)

In [ ]:
def answer_account(question: str, top_k: int = 4) -> AccountAnswer:
    """
    Account v1（structured_output 版）：
    1. 用（暂时复用的）faq_retriever 检索 top_k 段文本
    2. 拼成 relevant_content
    3. 调用 account_answer_chain，返回 AccountAnswer 对象
    """
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    relevant_content = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    result: AccountAnswer = account_answer_chain.invoke(
        {
            "question": question,
            "relevant_content": relevant_content,
        }
    )

    return result

In [ ]:
def account_node(state: SupportState) -> SupportState:
    """Account 处理节点：调用 answer_account，并把结果写回 state。"""
    question = state["query"]
    acc_result = answer_account(question)

    parts = [
        f"【账号/计费简要说明】\n{acc_result.short_answer}",
        f"\n【详细说明】\n{acc_result.details}",
    ]
    if acc_result.next_steps and acc_result.next_steps.strip() and acc_result.next_steps.strip() != "无":
        parts.append(f"\n【后续建议】\n{acc_result.next_steps}")

    state["answer"] = "\n".join(parts)
    # 如果你也想保存这次检索到的 docs，可以改 answer_account 返回 (result, docs)
    # 然后这里赋值 state["retrieved_docs"] = docs
    return state

#### test

In [ ]:
q1 = "免费额度用完之后，接口会发生什么？"
q2 = "为什么会报 quota exceeded 这种错误？一般怎么处理？"

for q in [q1, q2]:
    print("=" * 80)
    print("问题：", q)
    acc = answer_account(q)
    print("\n[short_answer]")
    print(acc.short_answer)
    print("\n[details]")
    print(acc.details)
    print("\n[next_steps]")
    print(acc.next_steps)
    print("\n\n")

In [ ]:
def other_node(state: SupportState) -> SupportState:
    """兜底节点：当意图不是 faq / tech_issue / account 时的回复。"""
    query = state["query"]
    state["answer"] = (
        "目前这个 TechSupport-Agent 主要支持以下几类问题：\n"
        "1) API / 参数 / 错误码等文档类问题（faq）\n"
        "2) 报错/环境/排错类问题（tech_issue）\n"
        "3) 账号 / 计费 / 配额类问题（account）\n\n"
        f"你刚才的问题暂时被归类为 'other'：\n\n"
        f"「{query}」\n\n"
        "你可以尝试：\n"
        "- 更具体地描述你调用的接口、报错信息\n"
        "- 说明你关注的是：用法？报错？还是配额/计费？"
    )
    return state

### graph骨架

##### 条件边

In [ ]:
def route_after_intent(state: SupportState) -> str:
    """
    根据 state['intent'] 决定下一步走哪个节点。
    返回值要跟 add_conditional_edges 里的 key 对得上。
    """
    intent = state.get("intent", "other")
    if intent not in ("faq", "tech_issue", "account", "other"):
        return "other"
    return intent

In [ ]:
# 1. 定义工作流，状态类型用 SupportState
workflow = StateGraph(SupportState)

# 2. 注册各个节点函数（你前面已经实现好了）
workflow.add_node("intent_classifier", intent_classifier_node)
workflow.add_node("faq", faq_node)
workflow.add_node("tech_issue", tech_issue_node)
workflow.add_node("account", account_node)
workflow.add_node("other", other_node)

# 3. 从 START 进图，先跑意图分类
workflow.set_entry_point("intent_classifier")

# 4. 条件路由函数
def route_after_intent(state: SupportState) -> str:
    intent = state.get("intent", "other")
    if intent not in ("faq", "tech_issue", "account", "other"):
        return "other"
    return intent

# 5. 从 intent_classifier 出发，根据 intent 去不同节点
workflow.add_conditional_edges(
    "intent_classifier",
    route_after_intent,
    {
        "faq": "faq",
        "tech_issue": "tech_issue",
        "account": "account",
        "other": "other",
    },
)


# 6. 告诉 LangGraph：这几个节点走完就是结束（连到 END）
workflow.add_edge("faq", END)
workflow.add_edge("tech_issue", END)
workflow.add_edge("account", END)
workflow.add_edge("other", END)

support_agent_app = workflow.compile()

### test

In [ ]:
q = "我调用 DeepSeek 的 chat 接口总是 401 unauthorized，怎么排查？"
result = support_agent_app.invoke({"query": q})
print("intent:", result.get("intent"))
print("answer:\n", result.get("answer", ""))
